# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}\nIdentifier: {getattr(metadata, 'identifier', 'N/A')}\n")
print(f"License: {metadata.license}\n")
print("Keywords:", getattr(metadata, 'keywords', []))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
print("Available Record Sets (@id and name):\n")
record_sets = list(dataset.metadata.record_sets)
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")

# For each record set, print its fields and columns by @id
for rs in record_sets:
    print(f"\nRecordSet: {rs.name} (@id: {rs.id})")
    print("  Fields:")
    for field in getattr(rs, 'fields', []):
        col = getattr(field, 'columns', [None])[0]
        col_id = col.id if col else 'N/A'
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type}, column: {col_id}")

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis.
Use the record set and field `@id`s from the overview above.

In [ ]:
# List all record set @id's
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

print("Loading records into DataFrames:")
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"- Loaded {len(df)} rows for record set {record_set_id}.")

# Show columns and sample for the first record set
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in main DataFrame ({main_record_set_id}):\n", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
Remove outliers, transform data, or group by key attributes.

In [ ]:
# Identify a suitable numeric field and group field for EDA
# We'll inspect the dataframe columns and choose one
main_df = dataframes[main_record_set_id]
print("First few columns:", main_df.columns[:10].tolist())

# For demonstration, use the first matching integer/float column
numeric_field = None
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field = col
        break
if numeric_field is None:
    # Try to coerce a likely column
    possible_cols = [c for c in main_df.columns if 'abundance' in c.lower() or 'count' in c.lower() or 'mw' in c.lower() or 'coverage' in c.lower() or 'peptide' in c.lower()]
    if possible_cols:
        numeric_field = possible_cols[0]
        main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')
        print(f"Coerced {numeric_field} to numeric.")
    else:
        print("No suitable numeric column found.")

# Choose a group field if available (e.g., 'description', 'accession', etc.)
group_field = None
for gf in ['description', 'accession', 'protein', 'group', 'sample']:
    if gf in main_df.columns:
        group_field = gf
        break

# Proceed with EDA if a numeric field is found
if numeric_field:
    threshold = main_df[numeric_field].mean() if main_df[numeric_field].notna().any() else 0
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (showing up to 5):")
    display(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records (showing up to 5):")
    display(filtered_df[[numeric_field, norm_col]].head())

    if group_field:
        print(f"\nGrouped data by {group_field} (showing means for first 5 groups):")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    if group_field:
        # Plot mean value by group (first 10 groups)
        grouped_df = main_df.groupby(group_field)[numeric_field].mean().reset_index().dropna().sort_values(numeric_field, ascending=False)
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df.head(10))
        plt.title(f"Mean {numeric_field} by {group_field} (top 10)")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The Mass Spectrometry dataset on extracellular vesicles from human mast cells contains extensive protein-level quantification and metadata.
- Metadata and structure can be programmatically explored with `mlcroissant`, using `@id` to reference all fields and record sets.
- Numeric analysis and visualization were performed on a selected quantitative column to reveal distribution patterns and group-level summaries.
- This notebook can be further extended with downstream analysis, domain-specific normalization, or integrated into ML pipelines.